In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import itertools
import seaborn as sns
import time
import pandas as pd
import cv2
import torchvision.transforms.functional as TF

In [5]:
ROOT_DIR = r"C:\Users\omarh\OneDrive - Georgia Institute of Technology\openEDS2019"

**Dataloader**

In [48]:
class EyeBoundingBoxDataset(Dataset):
    def __init__(self, subject_ids, root_dir, transform=None, apply_preprocessing=True, timing_enabled=False):
        self.root_dir = root_dir
        self.subject_ids = subject_ids
        self.transform = transform
        self.apply_preprocessing = apply_preprocessing
        self.timing_enabled = timing_enabled

        # For timing
        self.total_preprocessing_time = 0.0
        self.num_preprocessing_calls = 0

        self.data = []

        # Precompute Gamma LUT
        gamma = 0.8
        self.gamma_LUT = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)

        for subject_id in self.subject_ids:
            subject_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_id)
            bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_id}.txt")
            with open(bbox_file, 'r') as f:
                bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]
            for i in range(len(bboxes) - 1):
                img0_path = os.path.join(subject_dir, f"{i}.png")
                img1_path = os.path.join(subject_dir, f"{i+1}.png")
                self.data.append((img0_path, img1_path, bboxes[i+1]))

    def __len__(self):
        return len(self.data)

    def preprocess_image(self, img_path):
        start_time = time.perf_counter() if self.timing_enabled else None

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Could not load image from: {img_path}")

        img = cv2.LUT(img, self.gamma_LUT)

        p_low, p_high = np.percentile(img, (1, 99))
        lut_indices = np.arange(256)
        stretch_LUT = np.clip((lut_indices - p_low) * (255.0 / max(p_high - p_low, 1)), 0, 255).astype(np.uint8)
        img = cv2.LUT(img, stretch_LUT)

        if self.timing_enabled:
            end_time = time.perf_counter()
            self.total_preprocessing_time += (end_time - start_time) * 1000  # ms
            self.num_preprocessing_calls += 1

        return img

    def __getitem__(self, idx):
        img0_path, img1_path, bbox = self.data[idx]

        if self.apply_preprocessing:
            img0 = self.preprocess_image(img0_path)
            img1 = self.preprocess_image(img1_path)
        else:
            img0 = cv2.imread(img0_path, cv2.IMREAD_GRAYSCALE)
            img1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)

        # Resize manually because we skip torchvision transforms here
        img0 = cv2.resize(img0, (64, 64), interpolation=cv2.INTER_LINEAR)
        img1 = cv2.resize(img1, (64, 64), interpolation=cv2.INTER_LINEAR)

        img0 = torch.from_numpy(img0).unsqueeze(0).float() / 255.0
        img1 = torch.from_numpy(img1).unsqueeze(0).float() / 255.0

        diff = img1 - img0
        input_tensor = torch.cat((img1, diff), dim=0)

        target = torch.tensor(bbox, dtype=torch.float32)

        return input_tensor, target

    def get_average_preprocessing_time(self):
        if self.num_preprocessing_calls == 0:
            return 0.0
        return self.total_preprocessing_time / self.num_preprocessing_calls

**Model Definition**

In [7]:
class LightweightBBoxCNN(nn.Module):
    def __init__(self, hidden_size=64):  # now configurable
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, hidden_size),  # updated hidden size
            nn.ReLU(),
            nn.Linear(hidden_size, 4)  # 4 = xmin, xmax, ymin, ymax
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

**Split by Subject**

In [8]:
def split_subjects_and_save(root_dir, output_file='subject_split.txt', seed=42):
    random.seed(seed)
    subject_path = os.path.join(root_dir, 'openEDS', 'openEDS')
    all_subjects = sorted([d for d in os.listdir(subject_path) if d.startswith('S_') and os.path.isdir(os.path.join(subject_path, d))])

    random.shuffle(all_subjects)
    n_total = len(all_subjects)
    n_train = int(0.7 * n_total)
    n_val = int(0.2 * n_total)

    train_subjects = all_subjects[:n_train]
    val_subjects = all_subjects[n_train:n_train + n_val]
    test_subjects = all_subjects[n_train + n_val:]

    with open(output_file, 'w') as f:
        f.write("Training Subjects:\n")
        for s in train_subjects:
            f.write(f"{s}\n")
        f.write("\nValidation Subjects:\n")
        for s in val_subjects:
            f.write(f"{s}\n")
        f.write("\nTest Subjects:\n")
        for s in test_subjects:
            f.write(f"{s}\n")

    print(f"Subject split saved to {output_file}")
    return train_subjects, val_subjects, test_subjects

**Random Augmentations**

In [9]:
class RandomAugmentations:
    def __init__(self, p=0.2):
        self.p = p

    def __call__(self, x):
        # Always tensor input [C, H, W]
        if random.random() < self.p:
            # Horizontal Flip
            if random.random() < 0.5:
                x = TF.hflip(x)
            # Random Rotation
            if random.random() < self.p:
                angle = random.uniform(-45, 45)
                x = TF.rotate(x, angle, interpolation=transforms.InterpolationMode.BILINEAR)
            # Random Scaling
            if random.random() < self.p:
                scale_factor = random.uniform(0.8, 1.2)
                h, w = x.shape[1:]
                new_h, new_w = int(h * scale_factor), int(w * scale_factor)
                x = TF.resize(x, (new_h, new_w))
                x = TF.center_crop(x, (h, w))  # Keep original size after scaling
            # Gaussian Blur (from OpenCV)
            if random.random() < self.p:
                x = TF.gaussian_blur(x, kernel_size=7, sigma=random.uniform(2, 7))
            # Random Translation
            if random.random() < self.p:
                max_dx = 20
                max_dy = 20
                dx = random.randint(-max_dx, max_dx)
                dy = random.randint(-max_dy, max_dy)
                x = TF.affine(x, angle=0, translate=[dx, dy], scale=1, shear=[0, 0])
            # Image corruption with thin lines
            if random.random() < self.p:
                num_lines = random.randint(2, 9)
                x = self.draw_random_lines(x, num_lines)
        return x
    
    def draw_random_lines(self, x, num_lines):
        # x: tensor of shape [1, H, W]
        _, h, w = x.shape
        center_x = random.randint(0, w-1)
        center_y = random.randint(0, h-1)

        img_np = (x.squeeze(0).cpu().numpy() * 255).astype(np.uint8)
        for _ in range(num_lines):
            angle = random.uniform(0, 360)
            length = random.randint(10, 50)
            x1 = int(center_x + length * np.cos(np.deg2rad(angle)))
            y1 = int(center_y + length * np.sin(np.deg2rad(angle)))
            x1 = np.clip(x1, 0, w-1)
            y1 = np.clip(y1, 0, h-1)
            cv2.line(img_np, (center_x, center_y), (x1, y1), (255,), thickness=1)
        img_np = img_np / 255.0
        return torch.from_numpy(img_np).unsqueeze(0).float()

**Subject-Based Training**

In [10]:
def train_model_by_subject(root_dir, train_subjects, val_subjects, model=None, num_epochs=20, batch_size=16, lr=1e-3):
    # ---------------------
    # Augmentations for TRAINING
    # ---------------------
    train_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        RandomAugmentations(p=0.2),  # your requested augmentations
    ])

    # ---------------------
    # No augmentations for VALIDATION
    # ---------------------
    val_transform = transforms.Compose([
        transforms.Resize((64, 64)),
    ])

    train_dataset = EyeBoundingBoxDataset(train_subjects, root_dir, transform=train_transform, apply_preprocessing=True)
    val_dataset = EyeBoundingBoxDataset(val_subjects, root_dir, transform=val_transform, apply_preprocessing=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    device = next(model.parameters()).device if model else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if model is None:
        model = LightweightBBoxCNN().to(device)
    else:
        model.to(device)

    torch.autograd.set_detect_anomaly(True)

    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        avg_train_loss = train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item() * inputs.size(0)

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)

        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), f'checkpoints/ppaug_cp_e{epoch + 1}')
            print('Checkpoint saved...')

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

    return model, train_losses, val_losses

In [11]:
# Parameters
BATCH_SIZE = 4
HIDDEN_SIZE = 64
LEARNING_RATE = 0.01
NUM_EPOCHS = 50
SUBJECT_SPLIT_FILE = 'subject_split.txt'  # you can change this if you want

# 1. Split subjects and save split
train_subjects, val_subjects, test_subjects = split_subjects_and_save(ROOT_DIR, output_file=SUBJECT_SPLIT_FILE)

# 2. Initialize model with optimized hidden layer size
model = LightweightBBoxCNN(hidden_size=HIDDEN_SIZE)

# 3. Train the model (with training-time augmentations only)
model, train_losses, val_losses = train_model_by_subject(
    root_dir=ROOT_DIR,
    train_subjects=train_subjects,
    val_subjects=val_subjects,
    model=model,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE
)

Subject split saved to subject_split.txt
Epoch 1/50 - Train Loss: 27.3432 - Val Loss: 20.3230
Epoch 2/50 - Train Loss: 16.2177 - Val Loss: 14.3568
Epoch 3/50 - Train Loss: 13.8112 - Val Loss: 13.1150
Epoch 4/50 - Train Loss: 12.6542 - Val Loss: 13.2418
Checkpoint saved...
Epoch 5/50 - Train Loss: 12.0354 - Val Loss: 12.4027
Epoch 6/50 - Train Loss: 11.7473 - Val Loss: 12.9166
Epoch 7/50 - Train Loss: 11.5090 - Val Loss: 11.9892
Epoch 8/50 - Train Loss: 11.2030 - Val Loss: 12.7944
Epoch 9/50 - Train Loss: 11.0154 - Val Loss: 12.3080
Checkpoint saved...
Epoch 10/50 - Train Loss: 10.8852 - Val Loss: 13.0082
Epoch 11/50 - Train Loss: 11.0273 - Val Loss: 12.9331
Epoch 12/50 - Train Loss: 10.6562 - Val Loss: 11.7135
Epoch 13/50 - Train Loss: 10.4910 - Val Loss: 11.6054
Epoch 14/50 - Train Loss: 10.5501 - Val Loss: 12.6428
Checkpoint saved...
Epoch 15/50 - Train Loss: 10.5301 - Val Loss: 12.7598
Epoch 16/50 - Train Loss: 10.5402 - Val Loss: 12.1509
Epoch 17/50 - Train Loss: 10.6697 - Val Loss

**Test Model**

In [43]:
def compute_iou(boxA, boxB):
    xA1, xA2, yA1, yA2 = boxA
    xB1, xB2, yB1, yB2 = boxB

    x_left = max(xA1, xB1)
    y_top = max(yA1, yB1)
    x_right = min(xA2, xB2)
    y_bottom = min(yA2, yB2)

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    inter = (x_right - x_left) * (y_bottom - y_top)
    areaA = (xA2 - xA1) * (yA2 - yA1)
    areaB = (xB2 - xB1) * (yB2 - yB1)
    union = areaA + areaB - inter

    return inter / union if union > 0 else 0.0

In [44]:
def preload_test_data(dataset):
    """Preloads all raw data paths and bboxes into memory (without preprocessing yet)."""
    raw_data = []
    for img0_path, img1_path, bbox in tqdm(dataset.data, desc="Collecting test paths"):
        raw_data.append((img0_path, img1_path, bbox))
    return raw_data

In [45]:
def preprocess_images(img0_path, img1_path, preprocess_fn):
    """Loads and preprocesses image pair."""
    img0 = preprocess_fn(img0_path)
    img1 = preprocess_fn(img1_path)

    img0 = torch.from_numpy(img0).unsqueeze(0).float() / 255.0
    img1 = torch.from_numpy(img1).unsqueeze(0).float() / 255.0

    diff = img1 - img0
    input_tensor = torch.cat((img1, diff), dim=0)
    return input_tensor

In [ ]:
def test_real_time_model(model, root_dir, test_subjects, batch_size=1, device=None):
    # Choose device
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    # Move model to correct device
    model = model.to(device)
    model.eval()

    # Dataset with timing enabled
    test_dataset = EyeBoundingBoxDataset(
        subject_ids=test_subjects,
        root_dir=root_dir,
        transform=None,
        apply_preprocessing=True,
        timing_enabled=True  # ✅ Enable timing inside dataset
    )

    total_tensor_conversion_time = 0.0
    total_inference_time = 0.0
    all_ious = []
    num_samples = len(test_dataset)

    with torch.no_grad():
        for input_tensor, target in tqdm(test_dataset, desc="Testing model"):
            tensor_start = time.perf_counter()
            input_tensor = input_tensor.unsqueeze(0).to(device)  # Add batch dimension and move to device
            tensor_end = time.perf_counter()

            total_tensor_conversion_time += (tensor_end - tensor_start) * 1000  # ms

            infer_start = time.perf_counter()
            outputs = model(input_tensor)
            torch.cuda.synchronize() if device.type == 'cuda' else None
            infer_end = time.perf_counter()

            total_inference_time += (infer_end - infer_start) * 1000  # ms

            outputs = outputs.squeeze(0).cpu().numpy()
            target = target.numpy()

            iou = compute_iou(outputs, target)
            all_ious.append(iou)

    avg_preprocessing_time_ms = test_dataset.get_average_preprocessing_time()
    avg_tensor_conversion_time_ms = total_tensor_conversion_time / num_samples
    avg_inference_time_ms = total_inference_time / num_samples
    mean_iou = np.mean(all_ious)

    print(f"\nAverage Preprocessing Time (Gamma + Stretch): {avg_preprocessing_time_ms:.4f} ms")
    print(f"Average Tensor Conversion + Diff Time: {avg_tensor_conversion_time_ms:.4f} ms")
    print(f"Average Model Inference Time: {avg_inference_time_ms:.4f} ms")
    print(f"Mean IoU over Test Set: {mean_iou:.4f}")

    return avg_preprocessing_time_ms, avg_tensor_conversion_time_ms, avg_inference_time_ms, mean_iou

In [52]:
# Force CPU testing
avg_preproc_time_ms, avg_tensor_time_ms, avg_infer_time_ms, mean_iou = test_real_time_model(
    model=model,
    root_dir=ROOT_DIR,
    test_subjects=test_subjects,
    batch_size=1,
    device='cpu'  # 👈 manually force device
)

Testing model:   0%|          | 0/2792 [00:00<?, ?it/s]

Testing model: 100%|██████████| 2792/2792 [00:15<00:00, 178.28it/s]


Average Preprocessing Time (Gamma + Stretch): 2.5502 ms
Average Tensor Conversion + Diff Time: 0.0034 ms
Average Model Inference Time: 0.3474 ms
Mean IoU over Test Set: 0.8418
